# FixCheck — D4C Dataset

Runs FixCheck (codellama, 20 prefixes) on the D4C patches (GPT-4 generated) and compares against PReMS.

## 1. Upload inputs

Run this first so the files are ready while setup installs.

In [ ]:
from google.colab import files
print('Upload d4c_prms_results.csv and pred_d4c_gpt_archived.csv')
uploaded = files.upload()

Upload d4c_prms_results.csv and pred_d4c_gpt_archived.csv


Saving d4c_prms_results.csv to d4c_prms_results.csv
Saving pred_d4c_gpt_archived.csv to pred_d4c_gpt_archived.csv


## 2. Setup

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq openjdk-8-jdk openjdk-11-jdk openjdk-17-jdk \
    git subversion perl cpanminus ant locales wget curl unzip lsof zstd pciutils lshw
locale-gen en_US.UTF-8
echo 'Done'

(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../libperl5.38t64_5.38.2-3.2ubuntu0.6_amd64.deb ...
Unpacking libperl5.38t64:amd64 (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2ubuntu0.2) ...
Preparing to unpack .../perl_5.38.2-3.2ubuntu0.6_amd64.deb ...
Unpacking perl (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2ubuntu0.2) ...
Preparing to unpack .../perl-base_5.38.2-3.2ubuntu0.6_amd64.deb ...
Unpacking perl-base (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2ubuntu0.2) ...
Setting up perl-base (5.38.2-3.2ubuntu0.6) ...
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../000-perl-modules-5.38_5.38.2-3.2ubuntu0.6_all.deb ...
Unpacking perl-modules-5.38 (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2ubuntu0.2) ...
Selecting previously unselected package pci.ids.
Preparing to unpack .../001-pci.ids_0.0~2024.03.31-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Selecting previously unselected package libpci

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh
echo 'Ollama installed'

Ollama installed


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess, time, requests, os

subprocess.run(['pkill', '-f', 'ollama'], capture_output=True)
time.sleep(2)
ollama_env = {
    **os.environ,
    'LD_LIBRARY_PATH': '/usr/lib64-nvidia:/usr/local/cuda-12.8/targets/x86_64-linux/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
    'CUDA_VISIBLE_DEVICES': '0',
}
subprocess.Popen(['ollama', 'serve'],
    stdout=open('/tmp/ollama.log', 'w'),
    stderr=subprocess.STDOUT, env=ollama_env)
time.sleep(6)
try:
    r = requests.get('http://localhost:11434/api/tags')
    print('Ollama running:', r.status_code)
except Exception as e:
    print('ERROR:', e)
with open('/tmp/ollama.log') as f:
    log = f.read()
    if any(x in log for x in ['Tesla','CUDA','A100','L4']):
        print('GPU detected!')
    print(log[-300:])

Ollama running: 200
GPU detected!
i_id=0000:00:03.0 type=discrete total="22.0 GiB" available="21.8 GiB"
time=2026-10-14T23:53:09.583Z level=INFO source=routes.go:2551 msg="vram-based default context" total_vram="22.0 GiB" default_num_ctx=4096
[GIN] 2026/10/14 - 23:53:12 | 200 |     260.776µs |       127.0.0.1 | GET      "/api/tags"



In [ ]:
%%bash
ollama pull codellama
ollama list

NAME                ID              SIZE      MODIFIED               
codellama:latest    8fdf8f752f6e    3.8 GB    Less than a second ago    


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠏ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 3a43f

In [ ]:
%%bash
update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java
update-alternatives --set javac /usr/lib/jvm/java-17-openjdk-amd64/bin/javac
cd /content
git clone https://github.com/facumolina/fixcheck
cd fixcheck
./gradlew shadowJar
ls build/libs/

update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/java to provide /usr/bin/java (java) in manual mode
update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/javac to provide /usr/bin/javac (javac) in manual mode
...........10%............20%............30%............40%............50%............60%...........70%............80%............90%............100%

Welcome to Gradle 8.0.2!

Here are the highlights of this release:
 - Improvements to the Kotlin DSL
 - Fine-grained parallelism from the first build with configuration cache
 - Configurable Gradle user home cache cleanup

For more details see https://docs.gradle.org/8.0.2/release-notes.html

Starting a Gradle Daemon (subsequent builds will be faster)
> Task :compileJava
> Task :processResources NO-SOURCE
> Task :classes
> Task :shadowJar

BUILD SUCCESSFUL in 22s
2 actionable tasks: 2 executed


Cloning into 'fixcheck'...


In [ ]:
%%bash
pip install -q tokenizers==0.19.1
pip install -q pandas Flask==3.0.3 transformers==4.42.3 scikit-learn
echo 'Done'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.0/321.0 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 36.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 131.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 139.2 MB/s eta 0:00:00
Done


  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [ ]:
%%bash
cd /content
git clone https://github.com/rjust/defects4j
update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java
update-alternatives --set javac /usr/lib/jvm/java-8-openjdk-amd64/bin/javac
cpanm --notest DBI String::Interpolate
cd /content/defects4j
perl ./init.sh
echo 'Defects4J init done'

update-alternatives: using /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java to provide /usr/bin/java (java) in manual mode
update-alternatives: using /usr/lib/jvm/java-8-openjdk-amd64/bin/javac to provide /usr/bin/javac (javac) in manual mode
--> Working on DBI
Fetching http://www.cpan.org/authors/id/H/HM/HMBRAND/DBI-1.655.tgz ... OK
Configuring DBI-1.655 ... OK
Building DBI-1.655 ... OK
--> Working on String::Interpolate
Fetching http://www.cpan.org/authors/id/N/NE/NEILB/String-Interpolate-0.33.tar.gz ... OK
Configuring String-Interpolate-0.33 ... OK
==> Found dependencies: Safe::Hole, PadWalker
--> Working on Safe::Hole
Fetching http://www.cpan.org/authors/id/T/TO/TODDR/Safe-Hole-0.17.tar.gz ... OK
Configuring Safe-Hole-0.17 ... OK
Building Safe-Hole-0.17 ... OK
--> Working on PadWalker
Fetching http://www.cpan.org/authors/id/R/RO/ROBIN/PadWalker-2.5.tar.gz ... OK
Configuring PadWalker-2.5 ... OK
Building PadWalker-2.5 ... OK
Building String-Interpolate-0.33 ... OK
4 distributions inst

Cloning into 'defects4j'...
Updating files: 100% (10125/10125), done.
curl: (3) URL using bad/illegal format or missing URL
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 7119k  100 7119k    0     0  3528k      0  0:00:02  0:00:02 --:--:-- 3529k
curl: (3) URL using bad/illegal format or missing URL
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 17.0M  100 17.0M    0     0  18.3M      0 --:--:-- --:--:-- --:--:-- 18.3M
curl: (3) URL using bad/illegal format or missing URL
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --

In [ ]:
%%bash
sed -i '235s/.*/        # die ("Java 11 is required!\\n\\n"); # patched/' /content/defects4j/framework/core/Constants.pm
grep -n 'Java 11' /content/defects4j/framework/core/Constants.pm

235:        # die ("Java 11 is required!\n\n"); # patched


In [ ]:
import os, subprocess
os.environ['PATH'] = '/content/defects4j/framework/bin:/usr/lib/jvm/java-8-openjdk-amd64/jre/bin:' + os.environ['PATH']
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'
r = subprocess.run(['defects4j', 'info', '-p', 'Math'], capture_output=True, text=True)
print(r.stdout[:200])

Summary of configuration for Project: Math
--------------------------------------------------------------------------------
    Script dir: /content/defects4j/framework
      Base dir: /content/defect


## 3. Main loop

Resumes from Drive automatically - completed patches are skipped. Safe to re-run after a crash or disconnect.

In [ ]:
import os, re, shutil, subprocess, pandas as pd, time, requests
from pathlib import Path
from sklearn.metrics import roc_auc_score

os.chdir('/content')

subprocess.run(['pkill', '-f', 'ollama'], capture_output=True)
time.sleep(2)
ollama_env = {
    **os.environ,
    'LD_LIBRARY_PATH': '/usr/lib64-nvidia:/usr/local/cuda-12.8/targets/x86_64-linux/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
    'CUDA_VISIBLE_DEVICES': '0',
}
subprocess.Popen(['ollama', 'serve'],
    stdout=open('/tmp/ollama.log', 'w'),
    stderr=subprocess.STDOUT, env=ollama_env)
time.sleep(6)
try:
    r = requests.get('http://localhost:11434/api/tags')
    print('Ollama running:', r.status_code)
except Exception as e:
    print('Ollama ERROR:', e)

subprocess.run('update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java', shell=True)
subprocess.run('update-alternatives --set javac /usr/lib/jvm/java-8-openjdk-amd64/bin/javac', shell=True)
subprocess.run("sed -i '235s/.*/        # die (\"Java 11 is required!\\\\n\\\\n\"); # patched/' /content/defects4j/framework/core/Constants.pm", shell=True)

JAVA8     = '/usr/lib/jvm/java-8-openjdk-amd64/bin'
JAVA8_JRE = '/usr/lib/jvm/java-8-openjdk-amd64/jre/bin'
JAVA17    = '/usr/lib/jvm/java-17-openjdk-amd64/bin'
D4J       = '/content/defects4j/framework/bin'
BASE      = '/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin'

setup_env = {**os.environ, 'PATH': f'{JAVA8_JRE}:{JAVA8}:{D4J}:{BASE}', 'JAVA_HOME': '/usr/lib/jvm/java-8-openjdk-amd64'}
run_env   = {**os.environ, 'PATH': f'{JAVA17}:{D4J}:{BASE}', 'JAVA_HOME': '/usr/lib/jvm/java-17-openjdk-amd64', 'FIXCHECK': '/content/fixcheck'}

ANT_PROJECTS = {'Chart', 'Closure', 'Time', 'Mockito'}

r = subprocess.run('defects4j info -p Math', shell=True, capture_output=True, text=True, env=setup_env)
print('D4J check:', r.stdout[:50] if r.returncode==0 else r.stderr[:200])

prems_df = pd.read_csv('d4c_prms_results.csv')
prems_df = prems_df.drop_duplicates(subset='slug', keep='last')
prems_df = prems_df[prems_df['status'] == 'OK'].copy()
_n = len(prems_df)
prems_df = prems_df[pd.to_numeric(prems_df['prmsc'], errors='coerce').notna()].copy()
print(f'Dropped {_n - len(prems_df)} OK rows with non-numeric prmsc (e.g. NO_MUTANTS)')
prems_df = prems_df.reset_index(drop=True)
prems_df['project'] = prems_df['slug'].str.split('_').str[0]
prems_df['bug']     = prems_df['slug'].str.split('_').str[1]
print(f'OK patches: {len(prems_df)}')

patches_df = pd.read_csv('pred_d4c_gpt_archived.csv')
patches_dict = {r['slug']: r for _, r in patches_df.iterrows()}
print(f'D4C patches loaded: {len(patches_dict)}')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_BACKUP = '/content/drive/MyDrive/fixcheck-d4c-v2'
LOCAL_OUTPUT = '/content/fixcheck/fixcheck-output/defects-repairing'
TMP_BASE     = '/content/defects4j_checkouts/base'
CSV_PATH     = f'{DRIVE_BACKUP}/d4c_prems_vs_fixcheck.csv'

os.makedirs(DRIVE_BACKUP, exist_ok=True)
os.makedirs(LOCAL_OUTPUT, exist_ok=True)
os.makedirs(TMP_BASE, exist_ok=True)

restored = 0
for folder in os.listdir(DRIVE_BACKUP):
    src_report = f'{DRIVE_BACKUP}/{folder}/codellama/report.csv'
    if os.path.exists(src_report):
        dst_dir = f'{LOCAL_OUTPUT}/{folder}/codellama'
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(src_report, f'{dst_dir}/report.csv')
        restored += 1
print(f'Restored {restored} report.csv files from Drive')
done = [d for d in os.listdir(LOCAL_OUTPUT) if os.path.exists(f'{LOCAL_OUTPUT}/{d}/codellama/report.csv')]
print(f'Already done: {len(done)}')

def save_csv(results):
    rows = []
    results_dict = {r['patch_uid']: r for r in results}
    for _, row in prems_df.iterrows():
        patch_uid   = row['slug']
        report_path = f'{LOCAL_OUTPUT}/{patch_uid}/codellama/report.csv'
        res = results_dict.get(patch_uid)
        status = res['status'] if res else ('DONE' if os.path.exists(report_path) else 'NOT_RUN')
        if not os.path.exists(report_path):
            rows.append({'patch_uid': patch_uid, 'bug_id': patch_uid,
                        'label': int(row['label']), 'prmsc': row['prmsc'],
                        'fixcheck_status': status, 'fixcheck_failing': None, 'fixcheck_pred': None})
            continue
        try:
            report = pd.read_csv(report_path)
            failing = int(report['assertion_failing_prefixes'].sum())
            fixcheck_pred = 0 if failing > 0 else 1
        except:
            failing, fixcheck_pred = None, None
        rows.append({'patch_uid': patch_uid, 'bug_id': patch_uid,
                    'label': int(row['label']), 'prmsc': row['prmsc'],
                    'fixcheck_status': 'DONE', 'fixcheck_failing': failing, 'fixcheck_pred': fixcheck_pred})
    df = pd.DataFrame(rows)
    df.to_csv(CSV_PATH, index=False)
    assessable = df[df['fixcheck_pred'].notna()].copy()
    if len(assessable) > 1 and assessable['label'].nunique() > 1:
        try:
            auc_fc = roc_auc_score(1 - assessable['label'], assessable['fixcheck_failing'])
            auc_pr = roc_auc_score(1 - assessable['label'], 1 - assessable['prmsc'])
            print(f'  [CSV saved — DONE:{len(assessable)} | FixCheck AUC:{auc_fc:.3f} | PReMS AUC:{auc_pr:.3f}]')
        except:
            print(f'  [CSV saved — DONE:{len(assessable)}]')
    return df

def extract_method_signature(bug_code):
    for line in bug_code.strip().splitlines():
        s = line.strip()
        if s and not s.startswith('//') and not s.startswith('*'):
            return s
    return ''

def get_body_snippet(bug_code, start=4, length=8):
    lines = bug_code.strip().splitlines()
    body = []
    for line in lines[start:start+length]:
        s = line.strip()
        if s and not s.startswith('//') and not s.startswith('*'):
            body.append(s)
    return ' '.join(body)[:200]

def find_fqcn(project_dir, bug_code):
    project_dir = Path(project_dir)
    sig = extract_method_signature(bug_code)
    method_m = re.search(r'(?:public|protected|private|static|final|\s)+\s+[\w\[\]<>,\s]+\s+(\w+)\s*\(', sig)
    method_name = method_m.group(1) if method_m else None
    body_snippet_early = get_body_snippet(bug_code, start=1, length=5)
    body_snippet_deep  = get_body_snippet(bug_code, start=5, length=8)
    all_tokens = re.findall(r'\b[A-Za-z][A-Za-z0-9_]{7,}\b', bug_code)
    rare_tokens = [t for t in all_tokens if len(t) > 9][:5]
    src_dirs = [
        project_dir / 'src' / 'main' / 'java',
        project_dir / 'src' / 'java',
        project_dir / 'src',
        project_dir / 'source',
    ]
    candidates = []
    for src_dir in src_dirs:
        if not src_dir.exists(): continue
        for java_file in src_dir.rglob('*.java'):
            try:
                content = java_file.read_text(errors='ignore')
                score = 0
                if method_name and re.search(r'\b' + re.escape(method_name) + r'\s*\(', content): score += 1
                if sig.strip() in content: score += 3
                if body_snippet_early and body_snippet_early[:80] in content: score += 3
                if body_snippet_deep and body_snippet_deep[:80] in content: score += 4
                for tok in rare_tokens:
                    if tok in content: score += 1
                if score >= 3:
                    rel = java_file.relative_to(src_dir)
                    fqcn = str(rel).replace('/', '.').replace('\\', '.').removesuffix('.java')
                    candidates.append((score, fqcn, java_file))
            except: continue
    if not candidates: return None, None
    candidates.sort(reverse=True)
    return candidates[0][1], candidates[0][2]

def apply_patch(java_file, bug_code, fix_code):
    java_file = Path(java_file)
    with open(java_file, 'rb') as f:
        content_bytes = f.read()
    if b'\r\n' in content_bytes:
        content = content_bytes.replace(b'\r\n', b'\n').decode('utf-8', errors='ignore')
        java_file.write_text(content)
    content = java_file.read_text(errors='ignore')
    file_lines = content.splitlines()
    bug_lines_raw = bug_code.strip().splitlines()
    n = len(bug_lines_raw)
    if bug_code.strip() in content:
        java_file.write_text(content.replace(bug_code.strip(), fix_code.strip(), 1))
        return True
    bug_rstripped = [l.rstrip() for l in bug_lines_raw]
    for i in range(len(file_lines) - n + 1):
        if [l.rstrip() for l in file_lines[i:i+n]] == bug_rstripped:
            new = '\n'.join(file_lines[:i] + fix_code.strip().splitlines() + file_lines[i+n:]) + '\n'
            java_file.write_text(new)
            return True
    bug_stripped = [l.strip() for l in bug_lines_raw]
    for i in range(len(file_lines) - n + 1):
        if [l.strip() for l in file_lines[i:i+n]] == bug_stripped:
            new = '\n'.join(file_lines[:i] + fix_code.strip().splitlines() + file_lines[i+n:]) + '\n'
            java_file.write_text(new)
            return True
    sig = extract_method_signature(bug_code).strip()
    last = bug_lines_raw[-1].strip()
    for i in range(len(file_lines) - n + 1):
        if sig in file_lines[i] and file_lines[i+n-1].strip() == last:
            interior_bug  = [l.strip() for l in bug_lines_raw[1:-1]]
            interior_file = [l.strip() for l in file_lines[i+1:i+n-1]]
            if len(interior_bug) == len(interior_file):
                matches = sum(1 for a, b in zip(interior_bug, interior_file) if a == b)
                if matches / max(len(interior_bug), 1) >= 0.6:
                    new = '\n'.join(file_lines[:i] + fix_code.strip().splitlines() + file_lines[i+n:]) + '\n'
                    java_file.write_text(new)
                    return True
    return False

def get_trigger_tests(project_dir):
    try:
        result = subprocess.check_output(
            ['defects4j', 'export', '-p', 'tests.trigger'],
            cwd=project_dir, text=True, stderr=subprocess.DEVNULL).strip()
        classes = set()
        for line in result.splitlines():
            line = line.strip()
            if '::' in line: classes.add(line.split('::')[0])
            elif line: classes.add(line)
        return sorted(classes)
    except: return []

def detect_input_class(project_dir, trigger_tests, trigger_method_name=None):
    for trigger_class in trigger_tests:
        class_file = trigger_class.split('.')[-1] + '.java'
        for root, dirs, files in os.walk(project_dir):
            if class_file in files:
                try:
                    content = open(os.path.join(root, class_file)).read()
                    if trigger_method_name:
                        idx = content.find(f'void {trigger_method_name}')
                        if idx >= 0:
                            brace = content.find('{', idx)
                            if brace >= 0:
                                depth, body = 0, ''
                                for ch in content[brace:]:
                                    if ch == '{': depth += 1
                                    elif ch == '}': depth -= 1
                                    body += ch
                                    if depth == 0: break
                                content = body
                    if re.search(r'\b\d+\.\d+\b', content) or 'double' in content: return 'double'
                    if re.search(r'\b\d+L\b', content) or 'long' in content: return 'long'
                    if '"' in content or 'String' in content: return 'java.lang.String'
                    if re.search(r'\b\d+\b', content) or 'int' in content: return 'int'
                    if 'true' in content or 'false' in content or 'boolean' in content: return 'boolean'
                except: pass
    return 'java.lang.Object'

def get_classpaths(working_dir, project):
    if project == 'Closure': return 'build/classes:build/test', 'build/test', 'test'
    if project == 'Mockito': return 'build/classes/main:build/classes/test', 'build/classes/test', 'test'
    if project in ANT_PROJECTS: return 'build:build-tests', 'build-tests', 'tests'
    if os.path.isdir(os.path.join(working_dir, 'src/test/java')): tests_src = 'src/test/java'
    elif os.path.isdir(os.path.join(working_dir, 'src/test')): tests_src = 'src/test'
    else: tests_src = 'src/test/java'
    return 'target/classes:target/test-classes', 'target/test-classes', tests_src

INPUT_TYPES = ['int', 'double', 'long', 'java.lang.String', 'boolean', 'java.lang.Object']

def run_fixcheck(patch_uid, working_dir, trigger_class, trigger_method, main_dep, tests_build, tests_src, input_class):
    os.chdir('/content/fixcheck')
    subject_cp = ':'.join([os.path.join(working_dir, d) for d in main_dep.split(':')])
    test_classes_path = os.path.join(working_dir, tests_build)
    target_test_dir = os.path.join(working_dir, tests_src)
    failure_log = os.path.join(working_dir, 'failing_tests')
    with open(failure_log, 'w') as f:
        f.write(f'--- {trigger_class}\n')
        f.write(f'{trigger_class}::{trigger_method}\n')
    props_file = f'/tmp/fixcheck-{patch_uid}.properties'
    with open(props_file, 'w') as f:
        f.write(f'test-classes-path={test_classes_path}\n')
        f.write(f'test-class={trigger_class}\n')
        f.write(f'test-methods={trigger_method}\n')
        f.write(f'test-classes-src={target_test_dir}\n')
        f.write(f'test-failure-trace-log={failure_log}\n')
        f.write(f'inputs-class={input_class}\n')
        f.write(f'number-of-prefixes=20\n')
        f.write(f'assertion-generator=codellama\n')
    full_cp = f'build/libs/fixcheck-all-1.0.0.jar:{subject_cp}:{test_classes_path}'
    cmd = f'/usr/lib/jvm/java-17-openjdk-amd64/bin/java -cp {full_cp} org.imdea.fixcheck.FixCheck -p {props_file}'
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=600, env=run_env)
    output_dir = f'fixcheck-output/defects-repairing/{patch_uid}/codellama'
    os.makedirs(output_dir, exist_ok=True)
    for fname in ['report.csv', 'scores-failing-tests.csv', 'log.out']:
        src = f'fixcheck-output/{fname}'
        if os.path.exists(src): subprocess.run(f'mv {src} {output_dir}/', shell=True)
    for dname in ['failing-tests', 'passing-tests', 'non-compiling-tests']:
        src = f'fixcheck-output/{dname}'
        if os.path.exists(src): subprocess.run(f'mv {src} {output_dir}/', shell=True)
    return r.stdout + r.stderr

def backup_report(patch_uid):
    report_src = f'{LOCAL_OUTPUT}/{patch_uid}/codellama/report.csv'
    if os.path.exists(report_src):
        dst_dir = f'{DRIVE_BACKUP}/{patch_uid}/codellama'
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(report_src, f'{dst_dir}/report.csv')

def ensure_ollama():
    try:
        requests.get('http://localhost:11434/api/tags', timeout=3)
        return True
    except:
        print('  [Ollama down, restarting...]', end=' ', flush=True)
        subprocess.run(['pkill', '-f', 'ollama'], capture_output=True)
        time.sleep(2)
        subprocess.Popen(['ollama', 'serve'], stdout=open('/tmp/ollama.log', 'w'), stderr=subprocess.STDOUT, env=ollama_env)
        time.sleep(6)
        try:
            requests.get('http://localhost:11434/api/tags', timeout=3)
            print('[Ollama back up]', end=' ', flush=True)
            return True
        except:
            print('[Ollama FAILED]')
            return False

save_csv([])

results = []

for i, row in prems_df.iterrows():
    os.chdir('/content')
    slug    = row['slug']
    project = row['project']
    bug     = row['bug']
    label   = int(row['label'])

    report_file = f'{LOCAL_OUTPUT}/{slug}/codellama/report.csv'
    if os.path.exists(report_file):
        print(f'[{i+1}/{len(prems_df)}] SKIP {slug}')
        results.append({'patch_uid': slug, 'bug_id': slug, 'label': label, 'prmsc': row['prmsc'], 'status': 'SKIPPED'})
        continue

    print(f'[{i+1}/{len(prems_df)}] {slug}', end=' ', flush=True)
    ensure_ollama()

    patch = patches_dict.get(slug, {})
    bug_code = patch.get('bug', '') or ''
    fix_code = patch.get('fix', '') or ''
    if not bug_code or not fix_code:
        print('NO_CODE')
        results.append({'patch_uid': slug, 'bug_id': slug, 'label': label, 'prmsc': row['prmsc'], 'status': 'NO_CODE'})
        save_csv(results)
        continue

    bug_base = os.path.join(TMP_BASE, f'{project}{bug}b')
    if not os.path.isdir(bug_base):
        cmd = f'defects4j checkout -p {project} -v {bug}b -w {bug_base}'
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=300, env=setup_env)
        if r.returncode != 0:
            print('CHECKOUT_FAILED')
            results.append({'patch_uid': slug, 'bug_id': slug, 'label': label, 'prmsc': row['prmsc'], 'status': 'CHECKOUT_FAILED'})
            save_csv(results)
            continue
        os.chdir(bug_base)
        subprocess.run('defects4j compile', shell=True, capture_output=True, env=setup_env, timeout=300)

    trigger_tests = get_trigger_tests(bug_base)
    if not trigger_tests:
        print('NO_TRIGGER')
        results.append({'patch_uid': slug, 'bug_id': slug, 'label': label, 'prmsc': row['prmsc'], 'status': 'NO_TRIGGER'})
        save_csv(results)
        continue

    r2 = subprocess.run(['defects4j', 'export', '-p', 'tests.trigger'],
                        cwd=bug_base, capture_output=True, text=True, env=setup_env)
    trigger_line = r2.stdout.strip().split('\n')[0]
    if '::' in trigger_line:
        trigger_class, trigger_method = trigger_line.split('::', 1)
    else:
        trigger_class, trigger_method = trigger_line, 'test'
    trigger_class  = trigger_class.strip()
    trigger_method = trigger_method.strip()

    fqcn, java_file_base = find_fqcn(bug_base, bug_code)
    if not fqcn:
        print('FQCN_NOT_FOUND')
        results.append({'patch_uid': slug, 'bug_id': slug, 'label': label, 'prmsc': row['prmsc'], 'status': 'FQCN_NOT_FOUND'})
        save_csv(results)
        continue
    print(f'fqcn={fqcn.split(".")[-1]}', end=' ', flush=True)

    input_class = detect_input_class(bug_base, trigger_tests, trigger_method)
    print(f'input={input_class}', end=' ', flush=True)

    working_dir = f'/content/defects4j_checkouts/{slug}/{project}{bug}b'
    os.makedirs(os.path.dirname(working_dir), exist_ok=True)
    if not os.path.isdir(working_dir):
        shutil.copytree(bug_base, working_dir)

    java_file_work = Path(working_dir) / Path(java_file_base).relative_to(Path(bug_base))
    backup_file = str(java_file_work) + '.orig'
    shutil.copy2(java_file_work, backup_file)

    if not apply_patch(java_file_work, bug_code, fix_code):
        print('PATCH_FAILED')
        results.append({'patch_uid': slug, 'bug_id': slug, 'label': label, 'prmsc': row['prmsc'], 'status': 'PATCH_FAILED'})
        shutil.copy2(backup_file, java_file_work)
        shutil.rmtree(working_dir, ignore_errors=True)
        save_csv(results)
        continue

    os.chdir(working_dir)
    r_compile = subprocess.run('defects4j compile', shell=True, capture_output=True, text=True, env=setup_env, timeout=300)
    combined_compile = r_compile.stdout + r_compile.stderr
    if 'FAILED' in combined_compile or r_compile.returncode != 0:
        print('COMPILE_FAILED')
        results.append({'patch_uid': slug, 'bug_id': slug, 'label': label, 'prmsc': row['prmsc'], 'status': 'COMPILE_FAILED'})
        shutil.rmtree(working_dir, ignore_errors=True)
        save_csv(results)
        continue

    main_dep, tests_build, tests_src = get_classpaths(working_dir, project)
    print('RUN...', end=' ', flush=True)

    status = 'NO_LOCALS'
    for ic in INPUT_TYPES:
        try:
            combined = run_fixcheck(slug, working_dir, trigger_class, trigger_method,
                                   main_dep, tests_build, tests_src, ic)
        except subprocess.TimeoutExpired:
            print('TIMEOUT')
            subprocess.run('pkill -f "fixcheck-all" || true', shell=True)
            status = 'TIMEOUT'
            break

        if os.path.exists(report_file):
            print(f'DONE(input={ic})', end=' ', flush=True)
            status = 'DONE'
            backup_report(slug)
            print('(backed up)', end=' ', flush=True)
            break
        elif 'No locals of type' in combined:
            continue
        elif 'result" is null' in combined:
            print('TEST_CRASH')
            status = 'TEST_CRASH'
            break
        elif 'compile' in combined.lower() and 'failed' in combined.lower():
            print('COMPILE_FAILED')
            status = 'COMPILE_FAILED'
            break
        else:
            print('UNKNOWN_FAIL')
            print(f'  tail: {combined[-200:]}')
            status = 'UNKNOWN_FAIL'
            break
    else:
        print('NO_LOCALS(all types)')

    results.append({'patch_uid': slug, 'bug_id': slug, 'label': label, 'prmsc': row['prmsc'], 'status': status})
    shutil.rmtree(working_dir, ignore_errors=True)
    save_csv(results)

results_df = pd.DataFrame(results)
print('\n=== Final Summary ===')
print(results_df['status'].value_counts())
final_df = save_csv(results)

Ollama running: 200
D4J check: Summary of configuration for Project: Math
-------
Dropped 30 OK rows with non-numeric prmsc (e.g. NO_MUTANTS)
OK patches: 52
D4C patches loaded: 437
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Restored 26 report.csv files from Drive
Already done: 26
  [CSV saved — DONE:26]
[1/52] SKIP Math_41
[2/52] Time_18 fqcn=ZonedChronology input=int PATCH_FAILED
  [CSV saved — DONE:26]
[3/52] Time_16 fqcn=DateTimeFormatter input=java.lang.String RUN... UNKNOWN_FAIL
  tail: CheckProperties.setup(FixCheckProperties.java:129)
	at org.imdea.fixcheck.properties.FixCheckProperties.getPrefixes(FixCheckProperties.java:142)
	at org.imdea.fixcheck.FixCheck.main(FixCheck.java:89)

  [CSV saved — DONE:26]
[4/52] Chart_13 fqcn=GridArrangement input=double PATCH_FAILED
  [CSV saved — DONE:26]
[5/52] SKIP Chart_5
[6/52] SKIP Closure_58
[7/52] Closure_161 fqcn=PeepholeFoldConstants input=java.lang.

## 4. Results

In [ ]:
from sklearn.metrics import roc_auc_score, matthews_corrcoef, confusion_matrix
import pandas as pd

df = pd.read_csv(CSV_PATH)
a = df[df['fixcheck_pred'].notna()].copy()
a['label'] = pd.to_numeric(a['label'], errors='coerce')
a['prmsc'] = pd.to_numeric(a['prmsc'], errors='coerce')
a['fixcheck_failing'] = pd.to_numeric(a['fixcheck_failing'], errors='coerce')
a['fixcheck_pred'] = pd.to_numeric(a['fixcheck_pred'], errors='coerce')
a = a.dropna(subset=['label','prmsc','fixcheck_failing','fixcheck_pred'])

print(f'Total patches:  {len(df)}')
print(f'Assessable:     {len(a)}')
print(f'  overfitting (label=0): {(a["label"]==0).sum()}')
print(f'  correct     (label=1): {(a["label"]==1).sum()}')
print()
print(df['fixcheck_status'].value_counts())
print()

auc_fc = roc_auc_score(1 - a['label'], a['fixcheck_failing'])
auc_pr = roc_auc_score(1 - a['label'], 1 - a['prmsc'])
mcc_fc = matthews_corrcoef(a['label'], a['fixcheck_pred'])
mcc_pr = matthews_corrcoef(a['label'], (a['prmsc'] >= 0.5).astype(int))

print(f'FixCheck  AUC: {auc_fc:.3f}   MCC: {mcc_fc:.3f}')
print(f'PReMS     AUC: {auc_pr:.3f}   MCC: {mcc_pr:.3f}')
print()

cm = confusion_matrix(a['label'], a['fixcheck_pred'])
print('FixCheck confusion matrix (rows=actual, cols=predicted):')
print('            pred=0  pred=1')
print(f'actual=0 {cm[0,0]:8d}{cm[0,1]:8d}   (overfitting)')
print(f'actual=1 {cm[1,0]:8d}{cm[1,1]:8d}   (correct)')

Total patches:  52
Assessable:     29
  overfitting (label=0): 2
  correct     (label=1): 27

fixcheck_status
DONE            29
UNKNOWN_FAIL    14
PATCH_FAILED     5
TEST_CRASH       4
Name: count, dtype: int64

FixCheck  AUC: 0.620   MCC: 0.164
PReMS     AUC: 0.759   MCC: 0.463

FixCheck confusion matrix (rows=actual, cols=predicted):
            pred=0  pred=1
actual=0        1       1   (overfitting)
actual=1        6      21   (correct)


## 5. Download and free the GPU

In [ ]:
from google.colab import files, runtime
files.download(CSV_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Disconnecting runtime to free GPU...
